# 01 — LASH Model and Fair Benchmark Training

This notebook performs the **primary model-training experiment**. It keeps the forecasting question identical across model families:

- 168 hours of historical context;
- direct prediction of all 24 future hours;
- hourly rolling forecast origins;
- one common causal information set;
- historical-only weather in the headline benchmark;
- validation-only HPO and residual-gain selection;
- purged, held-out router calibration;
- frozen train+validation final refit followed by one-time test evaluation.

The notebook writes Excel workbooks for human-readable reporting and compressed prediction/model artifacts for reproducible statistical and XAI analyses.

## 1. Repository setup

Use `smoke` only to confirm that every model family and output path works. Use `paper` for the values reported in the manuscript. Smoke mode intentionally caps expensive tree lengths and neural epochs and therefore must never be reported as a scientific result.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# %pip install -r ../requirements.txt

## 2. Fixed paper configuration

All four datasets use the same modeling code. The two external BDG aggregates are not allowed to trigger test-informed changes to the architecture, feature set, or evaluation rule.

The HPO sampler seeds and final training seeds are intentionally separated conceptually: repeated HPO measures search variability, while repeated final refits measure stochastic training variability after hyperparameters are fixed.

In [ ]:
from lash_revision_core import (
    ExperimentConfig, BENCHMARK_MODELS, MODEL_SEARCH_DIMENSIONS,
    search_space_manifest, runtime_environment, run_full_benchmark,
    read_benchmark_workbooks,
)

RUN_PROFILE = "paper"   # change to "smoke" only for a wiring check

config = ExperimentConfig(
    data_root=DATA_DIR,
    output_root=OUTPUT_DIR,
    run_profile=RUN_PROFILE,
    weather_mode="historical_only",
    dataset_keys=("CLUSTER_1", "CLUSTER_2", "BDG_EDU", "BDG_DORM"),
    hpo_seeds=(42, 142, 242),
    final_refit_seeds=(42, 142, 242),
    primary_hpo_repeats=3,
    external_hpo_repeats=3,
    tree_horizon_jobs=4,
    tree_threads_per_model=1,
    save_models=True,
    resume=True,
)

assert config.weather_mode == "historical_only"
print("Models:", BENCHMARK_MODELS)
display(pd.DataFrame([runtime_environment()]))
display(search_space_manifest())

## 3. Benchmark architecture and fairness contract

The benchmark contains persistence baselines, a causal anchor, a linear residual expert, five ensemble-learning families, six standard neural architectures, and the proposed hybrid.

### Tree ensembles

RF, GBM, XGBoost, LightGBM, and CatBoost use the **same flattened origin-level causal information** and train one independent direct regressor for each horizon `h=1,...,24`. A single HPO trial proposes one model-family configuration and evaluates that configuration across all 24 horizons. This prevents a tree model from gaining access to information unavailable to the sequential models while preserving a causally valid direct multi-step design.

### Neural models

MLP, LSTM, GRU, CNN-LSTM, TCN, Transformer, and the LASH sequential expert predict residual trajectories relative to the same anchor. Every neural comparator receives the same historical and future-known covariate tensors.

### LASH

The proposed model consists of a leakage-safe seasonal anchor, a Ridge residual expert, a causal TCN–GRB sequential residual expert, and a validation-calibrated horizon-aware router. `GRB` denotes a gated residual block and is deliberately distinguished from `GRU`.

In [ ]:
# The complete implementations are in src/lash_revision_core.py so that the same
# classes are imported by all notebooks rather than copied into multiple cells.
import inspect
from lash_revision_core import GatedResidualBlock, LASHSequentialNet

print("GatedResidualBlock source:")
print(inspect.getsource(GatedResidualBlock))
print("LASHSequentialNet constructor signature:")
print(inspect.signature(LASHSequentialNet))


## 4. Hyperparameter budget audit

Search-space dimensionality determines the number of trials within a prespecified 30–60 trial range. All searches are repeated three times with independent TPE seeds. Optuna executes sequentially so that the search sequence is reproducible.

In [ ]:
budget_table = pd.DataFrame([
    {
        "model": model,
        "tuned_dimensions": MODEL_SEARCH_DIMENSIONS[model],
        "trials_per_repeat": config.hpo_trials(model),
        "hpo_repeats": config.hpo_repeat_count("CLUSTER_1"),
        "total_trials_primary_dataset": config.hpo_trials(model) * config.hpo_repeat_count("CLUSTER_1"),
    }
    for model in MODEL_SEARCH_DIMENSIONS
]).sort_values("model")
display(budget_table)

## 5. Run the complete benchmark

The runner is incremental. Each dataset saves HPO trials, selected settings, predictions, and model artifacts as soon as they become available. SQLite-backed Optuna studies allow interrupted runs to continue without discarding completed trials.

For a full publication run this cell can take many hours because every tree-ensemble HPO trial fits 24 direct horizon models and every HPO search is repeated three times. That computational cost is intentional and is recorded in the output artifacts.

In [ ]:
RESULTS = run_full_benchmark(config)
print("Completed datasets:", list(RESULTS))

## 6. Inspect the automatically generated Excel workbooks

Each dataset receives one workbook containing benchmark summary statistics, seed-level metrics, horizon metrics, origin-level losses, selected hyperparameters, repeated-HPO summaries, router calibration, the complete search-space manifest, and the frozen protocol metadata.

Full horizon-by-origin predictions are stored as compressed NPZ files rather than duplicated into Excel; Notebook 03 discovers them automatically from the same output directory.

In [ ]:
workbooks = read_benchmark_workbooks(OUTPUT_DIR)
for dataset_key, sheets in workbooks.items():
    print("=" * 18, dataset_key, "=" * 18)
    display(sheets["Benchmark_Summary"].head(20))


## 7. HPO convergence files

Every Optuna trial is retained. The post-training notebook converts these records into best-so-far convergence curves and summarizes the variability among independent HPO repeats. This prevents a single lucky search from being treated as evidence of robust tuning.

In [ ]:
for dataset_key in config.dataset_keys:
    table_dir = OUTPUT_DIR / "benchmark" / dataset_key / "tables"
    trial_files = sorted(table_dir.glob("*_hpo_trials.csv"))
    print(dataset_key, "HPO trial files:", len(trial_files))